# NiyamTrace-X — External Dataset Validation Notebook (Colab Python 3.13 Fixed)

This notebook is designed to strengthen the Scientific Reports paper with **independent external data**, beyond the internally generated 2,000-case multilingual holdout.

It evaluates three external resources:

1. **MLCL / Lost in Execution (ACL 2026)** — multilingual tool-calling robustness derived from BFCL, with Chinese, Hindi, and Igbo variants.
2. **MASSIVE** — >1M multilingual NLU utterances with 60 intents and 55 slot types across 52 languages.
3. **MTOP** — multilingual task-oriented semantic parsing with intent and slot annotations.

## What this notebook measures

### MLCL
- tool/function selection accuracy
- required argument-key accuracy
- parameter-value exact/normalized match
- full tool-call exact match
- language-mismatch / interface-convention errors
- per-language metrics

### MASSIVE / MTOP
- intent accuracy
- slot-label precision / recall / F1
- slot-value grounding accuracy
- exact frame match
- hallucinated-slot rate
- per-language metrics

### Paper claim discipline

This notebook is an **external semantic/tool-interface generalization study**. It does not automatically prove end-to-end NiyamTrace-X authorization safety. The final paper should distinguish:
- **internal authorization evidence** from the frozen NiyamTrace-X benchmark;
- **external semantic/tool-calling evidence** from MLCL/MASSIVE/MTOP.

The notebook exports all raw predictions, metrics, figures, confidence intervals, manifests, and a paper-ready results summary as one ZIP.

In [ ]:
# CELL 1 — SAFE DEPENDENCY SETUP FOR COLAB PYTHON 3.13

# IMPORTANT:
# Do NOT upgrade/install numpy, pandas, scipy, or scikit-learn in this live kernel.
# Replacing NumPy while Colab is running can leave compiled extensions loaded
# against another NumPy ABI and trigger `_blas_supports_fpe` errors.
# This notebook computes all required metrics directly, so sklearn is unnecessary.

import sys, subprocess, importlib.util

REQUIRED = {
    "datasets": "datasets>=3.2.0",
    "transformers": "transformers>=4.48.0",
    "accelerate": "accelerate>=1.3.0",
    "bitsandbytes": "bitsandbytes>=0.45.0",
    "matplotlib": "matplotlib>=3.8.0",
    "tqdm": "tqdm>=4.66.0",
    "openai": "openai>=1.60.0",
    "huggingface_hub": "huggingface_hub>=0.28.0",
}

missing = [
    req for module, req in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages only:")
    for x in missing:
        print(" -", x)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *missing,
    ])
else:
    print("✅ Required packages already available.")

print("\nPython:", sys.version)
print("✅ Safe dependency setup complete.")

In [ ]:
# CELL 2 — ENVIRONMENT SANITY CHECK + CONFIGURATION

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import os, re, json, math, time, random, hashlib, zipfile, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, DatasetDict

print("Python environment loaded")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)

# Catch a damaged runtime before any long work starts.
_test = np.asarray([1.0, 2.0, 3.0], dtype=float)
assert np.isclose(_test.mean(), 2.0)
_test_df = pd.DataFrame({"x": [1, 2, 3]})
assert int(_test_df["x"].sum()) == 6
print("✅ NumPy/Pandas ABI sanity check passed.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path("/content/NTX_EXTERNAL_DATASET_VALIDATION")
DATA = BASE / "data"
RAW = BASE / "raw_predictions"
RESULTS = BASE / "results"
FIGURES = BASE / "figures"
EXPORT = BASE / "export"

for p in [BASE, DATA, RAW, RESULTS, FIGURES, EXPORT]:
    p.mkdir(parents=True, exist_ok=True)

MODE = os.getenv("NTX_EXT_MODE", "PAPER").upper()
assert MODE in {"SMOKE", "PAPER", "FULL"}

LIMITS = {
    "SMOKE": {"massive_per_lang": 20, "mtop_per_lang": 20, "mlcl_per_lang": 20, "bootstrap": 500},
    "PAPER": {"massive_per_lang": 250, "mtop_per_lang": 250, "mlcl_per_lang": 150, "bootstrap": 5000},
    "FULL": {"massive_per_lang": 1000, "mtop_per_lang": 1000, "mlcl_per_lang": None, "bootstrap": 10000},
}[MODE]

MASSIVE_LANGS = ["en-US", "hi-IN", "te-IN"]
MTOP_LANGS = ["en_XX", "hi_XX"]
MLCL_LANG_HINTS = ["en", "hi", "zh", "ig"]

BACKEND = os.getenv("NTX_MODEL_BACKEND", "LOCAL_QWEN").upper()
LOCAL_MODEL = os.getenv("NTX_LOCAL_MODEL", "Qwen/Qwen2.5-3B-Instruct")

OPENAI_BASE_URL = os.getenv("NTX_OPENAI_BASE_URL", "").strip()
OPENAI_API_KEY = os.getenv("NTX_OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.getenv("NTX_OPENAI_MODEL", "").strip()

MAX_NEW_TOKENS = 384
TEMPERATURE = 0.0

print("\nMODE:", MODE)
print("LIMITS:", LIMITS)
print("BACKEND:", BACKEND)
print("LOCAL_MODEL:", LOCAL_MODEL if BACKEND == "LOCAL_QWEN" else "n/a")

### Important if the old notebook already failed

If you already ran the older install cell and saw `_blas_supports_fpe`, restart the Colab session once:

**Runtime → Restart session**

Then open this **FIXED** notebook and run from Cell 1 onward. Do not rerun the old installation cell in that session.

In [ ]:
# CELL 3 — DOWNLOAD MASSIVE AND MTOP

def safe_load_dataset(*args, **kwargs):
    try:
        return load_dataset(*args, **kwargs)
    except Exception as e:
        print("Primary load failed:", repr(e))
        kwargs.pop("trust_remote_code", None)
        return load_dataset(*args, **kwargs)

print("Downloading MASSIVE...")
massive_all = {}
for lang in MASSIVE_LANGS:
    try:
        # MASSIVE's config name is language/locale specific.
        ds = safe_load_dataset("AmazonScience/massive", lang, trust_remote_code=True)
        massive_all[lang] = ds
        print(lang, {k: len(v) for k, v in ds.items()})
    except Exception as e:
        print(f"⚠️ MASSIVE {lang} unavailable:", repr(e))

print("\nDownloading MTOP...")
try:
    mtop = safe_load_dataset("tasksource/mtop", "mtop", trust_remote_code=True)
    print({k: len(v) for k, v in mtop.items()})
except Exception as e:
    mtop = None
    print("⚠️ MTOP unavailable:", repr(e))

assert massive_all, "No MASSIVE configuration loaded."

In [ ]:
# CELL 4 — OBTAIN MLCL

# Official ACL 2026 paper artifact:
MLCL_REPO_WEB = "https://anonymous.4open.science/r/multilingual_robustness_tool_calling-CA44"
MLCL_DIR = DATA / "mlcl_repo"

def run_cmd(cmd, cwd=None, timeout=600):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, timeout=timeout)

def try_clone_mlcl():
    if MLCL_DIR.exists() and any(MLCL_DIR.rglob("*")):
        print("MLCL repo already present:", MLCL_DIR)
        return True

    clone_candidates = [
        MLCL_REPO_WEB + ".git",
        MLCL_REPO_WEB,
    ]
    for url in clone_candidates:
        try:
            if MLCL_DIR.exists():
                shutil.rmtree(MLCL_DIR)
            p = run_cmd(["git", "clone", "--depth", "1", url, str(MLCL_DIR)], timeout=300)
            if p.returncode == 0 and any(MLCL_DIR.rglob("*")):
                print("✅ MLCL cloned from", url)
                return True
            print("Clone failed:", p.stderr[-600:])
        except Exception as e:
            print("Clone exception:", repr(e))
    return False

mlcl_ready = try_clone_mlcl()

if not mlcl_ready:
    print("""
MLCL automatic clone did not succeed.

Fallback:
1. Open the ACL paper artifact:
   https://anonymous.4open.science/r/multilingual_robustness_tool_calling-CA44
2. Download the repository ZIP.
3. Upload it when the next cell asks for a file.

The notebook will continue with MASSIVE/MTOP even if MLCL is unavailable.
""")

In [ ]:
# CELL 5 — OPTIONAL MLCL ZIP UPLOAD FALLBACK

if not mlcl_ready:
    try:
        from google.colab import files
        print("Upload the MLCL repository ZIP now, or cancel to continue without MLCL.")
        uploaded = files.upload()
        zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
        if zip_names:
            zname = zip_names[0]
            up = Path("/content") / zname
            if not up.exists():
                up.write_bytes(uploaded[zname])
            MLCL_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(up) as z:
                z.extractall(MLCL_DIR)
            mlcl_ready = True
            print("✅ MLCL ZIP extracted.")
    except Exception as e:
        print("MLCL upload skipped/unavailable:", repr(e))

def inventory_files(root: Path):
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for p in root.rglob("*"):
        if p.is_file():
            rows.append({
                "path": str(p.relative_to(root)),
                "size_bytes": p.stat().st_size,
                "suffix": p.suffix.lower(),
            })
    return pd.DataFrame(rows)

if mlcl_ready:
    mlcl_inventory = inventory_files(MLCL_DIR)
    display(mlcl_inventory.head(100))
    mlcl_inventory.to_csv(RESULTS / "00_mlcl_file_inventory.csv", index=False)

In [ ]:
# CELL 6 — NORMALIZE MASSIVE

def sample_rows(ds, n):
    n = len(ds) if n is None else min(n, len(ds))
    # deterministic sample
    idx = np.random.default_rng(SEED).choice(len(ds), size=n, replace=False)
    return ds.select(sorted(map(int, idx)))

def get_label_name(feature, value):
    try:
        return feature.int2str(int(value))
    except Exception:
        return str(value)

massive_rows = []

for lang, dsd in massive_all.items():
    split = "test" if "test" in dsd else list(dsd.keys())[-1]
    ds = sample_rows(dsd[split], LIMITS["massive_per_lang"])

    intent_feature = dsd[split].features.get("intent")
    scenario_feature = dsd[split].features.get("scenario")

    for row in ds:
        massive_rows.append({
            "dataset": "MASSIVE",
            "language": lang,
            "text": row.get("utt", row.get("text", "")),
            "gold_intent": get_label_name(intent_feature, row.get("intent")),
            "gold_scenario": get_label_name(scenario_feature, row.get("scenario")),
            "gold_slots_raw": row.get("annot_utt", ""),
            "id": str(row.get("id", "")),
        })

massive_df = pd.DataFrame(massive_rows)
massive_df.to_csv(DATA / "massive_eval.csv", index=False)
print("MASSIVE eval rows:", len(massive_df))
display(massive_df.head())

In [ ]:
# CELL 7 — NORMALIZE MTOP

mtop_rows = []

if mtop is not None:
    split = "test" if "test" in mtop else list(mtop.keys())[-1]
    mdf = mtop[split].to_pandas()

    # Dataset field may be "lang" with values such as hi_XX/en_XX.
    lang_col = "lang" if "lang" in mdf.columns else ("language" if "language" in mdf.columns else None)

    if lang_col:
        mdf = mdf[mdf[lang_col].isin(MTOP_LANGS)].copy()

    for lang, g in mdf.groupby(lang_col if lang_col else lambda _: "unknown"):
        n = len(g) if LIMITS["mtop_per_lang"] is None else min(LIMITS["mtop_per_lang"], len(g))
        g = g.sample(n=n, random_state=SEED)
        for _, row in g.iterrows():
            mtop_rows.append({
                "dataset": "MTOP",
                "language": lang,
                "text": row.get("question", row.get("utterance", "")),
                "gold_intent": row.get("intent", ""),
                "gold_domain": row.get("domain", ""),
                "gold_spans": row.get("spans", ""),
                "gold_logical_form": row.get("logical_form", ""),
                "id": str(row.get("idx", "")),
            })

mtop_df = pd.DataFrame(mtop_rows)
if len(mtop_df):
    mtop_df.to_csv(DATA / "mtop_eval.csv", index=False)

print("MTOP eval rows:", len(mtop_df))
display(mtop_df.head() if len(mtop_df) else pd.DataFrame({"status":["MTOP not loaded"]}))

In [ ]:
# CELL 8 — DISCOVER AND NORMALIZE MLCL FILES

def read_any_table(p: Path):
    try:
        if p.suffix.lower() == ".jsonl":
            return pd.read_json(p, lines=True)
        if p.suffix.lower() == ".json":
            obj = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(obj, list):
                return pd.json_normalize(obj)
            if isinstance(obj, dict):
                # common wrappers
                for k in ["data", "examples", "items", "records"]:
                    if isinstance(obj.get(k), list):
                        return pd.json_normalize(obj[k])
                return pd.json_normalize([obj])
        if p.suffix.lower() == ".csv":
            return pd.read_csv(p)
        if p.suffix.lower() in {".parquet", ".pq"}:
            return pd.read_parquet(p)
    except Exception:
        return None
    return None

def score_mlcl_candidate(path: Path, df: pd.DataFrame):
    if df is None or df.empty:
        return -1
    cols = " ".join(map(str.lower, df.columns))
    path_l = str(path).lower()
    score = 0
    for kw in ["question", "query", "prompt", "function", "tool", "answer", "ground", "language", "translated"]:
        if kw in cols:
            score += 1
    for kw in ["mlcl", "translated", "hindi", "igbo", "chinese", "bfcl"]:
        if kw in path_l:
            score += 1
    return score

mlcl_tables = []

if mlcl_ready:
    candidates = []
    for p in MLCL_DIR.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".jsonl",".json",".csv",".parquet",".pq"}:
            df = read_any_table(p)
            if df is not None and 0 < len(df) < 5_000_000:
                candidates.append((score_mlcl_candidate(p, df), p, df))

    candidates.sort(key=lambda x: x[0], reverse=True)

    print("Top MLCL candidate tables:")
    for score, p, df in candidates[:20]:
        print(score, p.relative_to(MLCL_DIR), df.shape, list(df.columns)[:15])

    # Preserve the top useful tables. The exact MLCL repository schema can evolve,
    # so downstream parsing uses field heuristics instead of hard-coding one filename.
    mlcl_tables = candidates[:20]

    inventory = []
    for score, p, df in mlcl_tables:
        inventory.append({
            "score": score,
            "path": str(p.relative_to(MLCL_DIR)),
            "rows": len(df),
            "columns": json.dumps(list(map(str, df.columns))),
        })
    pd.DataFrame(inventory).to_csv(RESULTS / "01_mlcl_candidate_tables.csv", index=False)
else:
    print("MLCL unavailable; MASSIVE + MTOP will still run.")

In [ ]:
# CELL 9 — LOAD MODEL BACKEND

_model = None
_tokenizer = None
_openai_client = None

if BACKEND == "LOCAL_QWEN":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    print("Loading local model:", LOCAL_MODEL)

    quant_config = None
    if torch.cuda.is_available():
        try:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        except Exception:
            quant_config = None

    _tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL, trust_remote_code=True)

    kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
    }
    if quant_config is not None:
        kwargs["quantization_config"] = quant_config
    else:
        kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32

    _model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, **kwargs)
    _model.eval()
    print("✅ Local model loaded.")

elif BACKEND == "OPENAI_COMPATIBLE":
    from openai import OpenAI
    if not (OPENAI_BASE_URL and OPENAI_MODEL):
        raise RuntimeError("Set NTX_OPENAI_BASE_URL and NTX_OPENAI_MODEL.")
    if not OPENAI_API_KEY:
        from getpass import getpass
        OPENAI_API_KEY = getpass("API key (or EMPTY for local vLLM): ").strip() or "EMPTY"
    _openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    print("✅ OpenAI-compatible backend configured:", OPENAI_MODEL)

else:
    raise ValueError("Unknown BACKEND: " + BACKEND)

In [ ]:
# CELL 10 — MODEL CALL + JSON PARSING HELPERS

def extract_json(text: str):
    text = text.strip()
    # remove common fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # first object
    start = text.find("{")
    if start < 0:
        return None

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    chunk = text[start:i+1]
                    try:
                        return json.loads(chunk)
                    except Exception:
                        return None
    return None

def call_llm(system: str, user: str, max_new_tokens=MAX_NEW_TOKENS):
    if BACKEND == "OPENAI_COMPATIBLE":
        r = _openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role":"system","content":system},
                {"role":"user","content":user},
            ],
            temperature=TEMPERATURE,
            max_tokens=max_new_tokens,
        )
        return r.choices[0].message.content or ""

    import torch
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":user},
    ]
    prompt = _tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = _tokenizer(prompt, return_tensors="pt").to(_model.device)

    with torch.inference_mode():
        out = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=_tokenizer.eos_token_id,
        )

    gen = out[0, inputs["input_ids"].shape[1]:]
    return _tokenizer.decode(gen, skip_special_tokens=True)

In [ ]:
# CELL 11 — SLOT PARSERS + METRICS

SLOT_RE = re.compile(r"\[([^:\]]+):([^\]]+)\]")

def parse_massive_slots(annot_utt: str):
    # MASSIVE annotated utterances commonly use [slot_name : value]-style markup.
    # We support several formatting variants conservatively.
    if not isinstance(annot_utt, str):
        return []
    pairs = []

    # [slot : value]
    for m in re.finditer(r"\[\s*([^:\]]+?)\s*:\s*([^\]]+?)\s*\]", annot_utt):
        pairs.append((m.group(1).strip(), m.group(2).strip()))

    # fallback: [SL:TYPE value]
    if not pairs:
        for m in re.finditer(r"\[\s*(?:SL:)?([A-Za-z0-9_]+)\s+([^\]]+?)\s*\]", annot_utt):
            pairs.append((m.group(1).strip(), m.group(2).strip()))
    return pairs

def parse_mtop_spans(spans: str, text: str):
    if not isinstance(spans, str) or not spans.strip():
        return []
    out = []
    for item in spans.split(","):
        item = item.strip()
        m = re.match(r"(\d+):(\d+):(?:SL:)?(.+)", item)
        if not m:
            continue
        a,b,label = int(m.group(1)),int(m.group(2)),m.group(3).strip()
        value = text[a:b] if 0 <= a <= b <= len(text) else ""
        out.append((label, value))
    return out

def normalize_text(x):
    x = "" if x is None else str(x)
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[\"'`]", "", x)
    return x

def slot_set(slots):
    return {(normalize_text(k), normalize_text(v)) for k,v in slots}

def label_set(slots):
    return {normalize_text(k) for k,_ in slots}

def prf(gold, pred):
    gold, pred = set(gold), set(pred)
    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)
    p = tp/(tp+fp) if tp+fp else (1.0 if not gold else 0.0)
    r = tp/(tp+fn) if tp+fn else 1.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p,r,f

def bootstrap_ci(values, n_boot=None, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values)==0:
        return (np.nan,np.nan,np.nan)
    n_boot = n_boot or LIMITS["bootstrap"]
    rng = np.random.default_rng(SEED)
    reps = np.empty(n_boot)
    for i in range(n_boot):
        reps[i] = rng.choice(values, size=len(values), replace=True).mean()
    return (
        values.mean(),
        np.quantile(reps, alpha/2),
        np.quantile(reps, 1-alpha/2),
    )

In [ ]:
# CELL 12 — MASSIVE EXTERNAL SEMANTIC EVALUATION

MASSIVE_SYSTEM = """You are evaluating multilingual task-oriented NLU.
Return ONLY JSON with this schema:
{
  "intent": "one exact intent label from the supplied list",
  "slots": [{"label":"slot_label","value":"surface value from the utterance"}]
}
Rules:
- Never invent a slot value that is not supported by the utterance.
- Keep names, numbers, dates, times, and locations grounded in the user's text.
- Use only labels from the supplied label lists.
- Do not add explanation.
"""

# Derive allowed labels from selected evaluation rows.
massive_intents = sorted(massive_df["gold_intent"].dropna().astype(str).unique())

# derive slot labels from gold
massive_slot_labels = sorted({
    k for annot in massive_df["gold_slots_raw"].fillna("")
    for k,_ in parse_massive_slots(annot)
})

print("MASSIVE intents:", len(massive_intents))
print("MASSIVE slot labels observed:", len(massive_slot_labels))

massive_pred_path = RAW / "massive_predictions.jsonl"
massive_done = {}

if massive_pred_path.exists():
    for line in massive_pred_path.read_text(encoding="utf-8").splitlines():
        try:
            r=json.loads(line); massive_done[r["row_key"]]=r
        except Exception:
            pass

with massive_pred_path.open("a", encoding="utf-8") as fout:
    for idx,row in tqdm(massive_df.iterrows(), total=len(massive_df), desc="MASSIVE"):
        row_key=f"{row['language']}::{row['id']}::{idx}"
        if row_key in massive_done:
            continue

        user = f"""Language: {row['language']}
Utterance: {row['text']}

Allowed intent labels:
{json.dumps(massive_intents, ensure_ascii=False)}

Allowed slot labels:
{json.dumps(massive_slot_labels, ensure_ascii=False)}
"""
        raw = call_llm(MASSIVE_SYSTEM,user)
        obj = extract_json(raw) or {}
        pred_slots = obj.get("slots", [])
        if not isinstance(pred_slots,list):
            pred_slots=[]

        clean_slots=[]
        for s in pred_slots:
            if isinstance(s,dict):
                clean_slots.append((str(s.get("label","")),str(s.get("value",""))))

        gold_slots=parse_massive_slots(row["gold_slots_raw"])
        gp,gr,gf=prf(slot_set(gold_slots),slot_set(clean_slots))
        lp,lr,lf=prf(label_set(gold_slots),label_set(clean_slots))

        grounded = [
            normalize_text(v) in normalize_text(row["text"])
            for _,v in clean_slots if normalize_text(v)
        ]

        rec={
            "row_key":row_key,
            "dataset":"MASSIVE",
            "language":row["language"],
            "id":row["id"],
            "text":row["text"],
            "gold_intent":row["gold_intent"],
            "pred_intent":str(obj.get("intent","")),
            "intent_correct": normalize_text(obj.get("intent",""))==normalize_text(row["gold_intent"]),
            "gold_slots":gold_slots,
            "pred_slots":clean_slots,
            "slot_pair_precision":gp,
            "slot_pair_recall":gr,
            "slot_pair_f1":gf,
            "slot_label_precision":lp,
            "slot_label_recall":lr,
            "slot_label_f1":lf,
            "all_pred_values_surface_grounded": all(grounded) if grounded else True,
            "hallucinated_slot_value_rate": 1-(sum(grounded)/len(grounded)) if grounded else 0.0,
            "frame_exact": (
                normalize_text(obj.get("intent",""))==normalize_text(row["gold_intent"])
                and slot_set(gold_slots)==slot_set(clean_slots)
            ),
            "raw_output":raw,
        }
        fout.write(json.dumps(rec,ensure_ascii=False)+"\n")
        fout.flush()

print("✅ MASSIVE predictions complete.")

In [ ]:
# CELL 13 — MTOP EXTERNAL SEMANTIC EVALUATION

if len(mtop_df):

    MTOP_SYSTEM = """You are evaluating multilingual task-oriented semantic parsing.
Return ONLY JSON:
{
  "intent": "exact allowed intent label",
  "slots": [{"label":"slot_label","value":"surface span from the utterance"}]
}
Rules:
- Do not invent unsupported entities.
- Slot values must be grounded in the utterance whenever the gold task expects a surface span.
- Use only supplied intent and slot labels.
"""

    mtop_intents=sorted(mtop_df["gold_intent"].dropna().astype(str).unique())
    mtop_slot_labels=sorted({
        k for _,r in mtop_df.iterrows()
        for k,_ in parse_mtop_spans(r["gold_spans"],r["text"])
    })

    mtop_pred_path=RAW/"mtop_predictions.jsonl"
    done={}
    if mtop_pred_path.exists():
        for line in mtop_pred_path.read_text(encoding="utf-8").splitlines():
            try:
                r=json.loads(line); done[r["row_key"]]=r
            except Exception:
                pass

    with mtop_pred_path.open("a",encoding="utf-8") as fout:
        for idx,row in tqdm(mtop_df.iterrows(),total=len(mtop_df),desc="MTOP"):
            row_key=f"{row['language']}::{row['id']}::{idx}"
            if row_key in done:
                continue

            user=f"""Language: {row['language']}
Utterance: {row['text']}
Allowed intent labels:
{json.dumps(mtop_intents,ensure_ascii=False)}
Allowed slot labels:
{json.dumps(mtop_slot_labels,ensure_ascii=False)}
"""
            raw=call_llm(MTOP_SYSTEM,user)
            obj=extract_json(raw) or {}
            ps=obj.get("slots",[])
            if not isinstance(ps,list): ps=[]
            pred_slots=[]
            for s in ps:
                if isinstance(s,dict):
                    pred_slots.append((str(s.get("label","")),str(s.get("value",""))))

            gold_slots=parse_mtop_spans(row["gold_spans"],row["text"])
            gp,gr,gf=prf(slot_set(gold_slots),slot_set(pred_slots))
            lp,lr,lf=prf(label_set(gold_slots),label_set(pred_slots))
            grounded=[
                normalize_text(v) in normalize_text(row["text"])
                for _,v in pred_slots if normalize_text(v)
            ]

            rec={
                "row_key":row_key,
                "dataset":"MTOP",
                "language":row["language"],
                "id":row["id"],
                "text":row["text"],
                "gold_intent":row["gold_intent"],
                "pred_intent":str(obj.get("intent","")),
                "intent_correct":normalize_text(obj.get("intent",""))==normalize_text(row["gold_intent"]),
                "gold_slots":gold_slots,
                "pred_slots":pred_slots,
                "slot_pair_precision":gp,
                "slot_pair_recall":gr,
                "slot_pair_f1":gf,
                "slot_label_precision":lp,
                "slot_label_recall":lr,
                "slot_label_f1":lf,
                "all_pred_values_surface_grounded":all(grounded) if grounded else True,
                "hallucinated_slot_value_rate":1-(sum(grounded)/len(grounded)) if grounded else 0.0,
                "frame_exact":(
                    normalize_text(obj.get("intent",""))==normalize_text(row["gold_intent"])
                    and slot_set(gold_slots)==slot_set(pred_slots)
                ),
                "raw_output":raw,
            }
            fout.write(json.dumps(rec,ensure_ascii=False)+"\n")
            fout.flush()

    print("✅ MTOP predictions complete.")
else:
    print("MTOP not available; skipping.")

In [ ]:
# CELL 14 — MLCL SCHEMA DISCOVERY + EVALUATION PREPARATION

def first_existing(row, names, default=None):
    for n in names:
        if n in row and row[n] is not None:
            return row[n]
    return default

def infer_language_from_path_and_row(path, row):
    text=(str(path)+" "+json.dumps(row,ensure_ascii=False,default=str)).lower()
    mapping=[
        ("hindi","hi"),("hi_","hi"),("chinese","zh"),("zh_","zh"),
        ("igbo","ig"),("ig_","ig"),("english","en"),("en_","en"),
    ]
    for token,lang in mapping:
        if token in text:
            return lang
    return "unknown"

def normalize_tool_definition(x):
    if isinstance(x,dict):
        return x
    if isinstance(x,list):
        return x
    if isinstance(x,str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def mlcl_records_from_candidates(candidates):
    out=[]
    for score,path,df in candidates:
        if score < 2:
            continue
        for _,r in df.iterrows():
            rd=r.to_dict()
            query=first_existing(rd,[
                "question","query","user_query","prompt","input","utterance","user"
            ])
            tools=first_existing(rd,[
                "function","functions","tools","tool","function_list","tool_list"
            ])
            gold=first_existing(rd,[
                "answer","answers","ground_truth","ground_truths","expected",
                "reference","target","tool_call","tool_calls"
            ])
            if query is None or tools is None or gold is None:
                continue
            out.append({
                "source_file":str(path.relative_to(MLCL_DIR)),
                "language":infer_language_from_path_and_row(path,rd),
                "query":query,
                "tools":normalize_tool_definition(tools),
                "gold":normalize_tool_definition(gold),
                "raw_row":rd,
            })
    return out

mlcl_records = mlcl_records_from_candidates(mlcl_tables) if mlcl_tables else []
print("Discovered MLCL-like records:",len(mlcl_records))

if mlcl_records:
    mdf=pd.DataFrame(mlcl_records)
    display(mdf[["source_file","language","query"]].head(20))

    # Deterministic per-language cap.
    picked=[]
    for lang,g in mdf.groupby("language"):
        n=len(g) if LIMITS["mlcl_per_lang"] is None else min(LIMITS["mlcl_per_lang"],len(g))
        picked.append(g.sample(n=n,random_state=SEED))
    mlcl_eval_df=pd.concat(picked,ignore_index=True)
    mlcl_eval_df.to_json(DATA/"mlcl_eval_records.jsonl",orient="records",lines=True,force_ascii=False)
    print("MLCL eval rows:",len(mlcl_eval_df),mlcl_eval_df["language"].value_counts().to_dict())
else:
    mlcl_eval_df=pd.DataFrame()
    print("""
⚠️ The notebook could not infer the MLCL table schema automatically.
Inspect results/01_mlcl_candidate_tables.csv and map query/tools/gold columns in this cell.
MASSIVE/MTOP remain fully evaluable.
""")

In [ ]:
# CELL 15 — MLCL TOOL-CALL EVALUATION

MLCL_SYSTEM = """You are a tool-calling model.
Given a user query and candidate tools, produce ONLY JSON:
{
  "function": "selected function name",
  "arguments": {"parameter":"value"}
}
Use the supplied tool schema exactly.
Do not translate function names or parameter keys.
Use execution-compatible argument values.
"""

def canonical_obj(x):
    if isinstance(x,str):
        try:
            x=json.loads(x)
        except Exception:
            return normalize_text(x)
    if isinstance(x,dict):
        return {str(k):canonical_obj(v) for k,v in sorted(x.items(),key=lambda kv:str(kv[0]))}
    if isinstance(x,list):
        return [canonical_obj(v) for v in x]
    return x

def find_function_and_args(obj):
    if isinstance(obj,str):
        try: obj=json.loads(obj)
        except Exception: return ("",{})
    if isinstance(obj,list) and obj:
        return find_function_and_args(obj[0])
    if not isinstance(obj,dict):
        return ("",{})
    # common formats
    if "function" in obj and isinstance(obj["function"],dict):
        fn=obj["function"].get("name","")
        args=obj["function"].get("arguments",{})
        return str(fn), normalize_tool_definition(args) if isinstance(args,str) else args
    fn=first_existing(obj,["function","name","tool_name","function_name"],"")
    args=first_existing(obj,["arguments","args","parameters","kwargs"],{})
    return str(fn), normalize_tool_definition(args) if isinstance(args,str) else args

if len(mlcl_eval_df):
    mlcl_pred_path=RAW/"mlcl_predictions.jsonl"
    done={}
    if mlcl_pred_path.exists():
        for line in mlcl_pred_path.read_text(encoding="utf-8").splitlines():
            try:
                r=json.loads(line); done[r["row_key"]]=r
            except Exception:
                pass

    with mlcl_pred_path.open("a",encoding="utf-8") as fout:
        for idx,row in tqdm(mlcl_eval_df.iterrows(),total=len(mlcl_eval_df),desc="MLCL"):
            row_key=f"{row['source_file']}::{idx}"
            if row_key in done:
                continue

            user=f"""Language: {row['language']}
User query:
{row['query']}

Candidate tools:
{json.dumps(row['tools'],ensure_ascii=False,default=str)}
"""
            raw=call_llm(MLCL_SYSTEM,user)
            pred=extract_json(raw) or {}
            gold=row["gold"]

            pfn,pargs=find_function_and_args(pred)
            gfn,gargs=find_function_and_args(gold)

            if not isinstance(pargs,dict): pargs={}
            if not isinstance(gargs,dict): gargs={}

            key_exact=set(map(str,pargs.keys()))==set(map(str,gargs.keys()))
            shared=set(map(str,pargs.keys())) & set(map(str,gargs.keys()))

            value_scores=[]
            for k in shared:
                value_scores.append(
                    normalize_text(pargs.get(k))==normalize_text(gargs.get(k))
                )

            full_exact=(
                normalize_text(pfn)==normalize_text(gfn)
                and canonical_obj(pargs)==canonical_obj(gargs)
            )

            rec={
                "row_key":row_key,
                "dataset":"MLCL",
                "source_file":row["source_file"],
                "language":row["language"],
                "query":row["query"],
                "gold_function":gfn,
                "pred_function":pfn,
                "function_correct":normalize_text(pfn)==normalize_text(gfn),
                "gold_arguments":gargs,
                "pred_arguments":pargs,
                "argument_key_exact":key_exact,
                "shared_argument_value_accuracy":(
                    float(np.mean(value_scores)) if value_scores else
                    (1.0 if not gargs and not pargs else 0.0)
                ),
                "full_tool_call_exact":full_exact,
                "raw_output":raw,
            }
            fout.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")
            fout.flush()

    print("✅ MLCL predictions complete.")
else:
    print("MLCL not available / schema not inferred — skipped.")

In [ ]:
# CELL 16 — AGGREGATE METRICS + BOOTSTRAP CONFIDENCE INTERVALS

def load_jsonl(path):
    if not path.exists():
        return pd.DataFrame()
    rows=[]
    for line in path.read_text(encoding="utf-8").splitlines():
        try: rows.append(json.loads(line))
        except Exception: pass
    return pd.DataFrame(rows)

massive_pred=load_jsonl(RAW/"massive_predictions.jsonl")
mtop_pred=load_jsonl(RAW/"mtop_predictions.jsonl")
mlcl_pred=load_jsonl(RAW/"mlcl_predictions.jsonl")

metric_rows=[]

def add_binary_metrics(df,dataset,group_col,metrics):
    if df.empty: return
    groups=[("ALL",df)] + list(df.groupby(group_col))
    for group,g in groups:
        for m in metrics:
            vals=pd.to_numeric(g[m],errors="coerce").dropna().to_numpy(float)
            mean,lo,hi=bootstrap_ci(vals)
            metric_rows.append({
                "dataset":dataset,
                "group":group,
                "metric":m,
                "n":len(vals),
                "mean":mean,
                "ci95_low":lo,
                "ci95_high":hi,
            })

add_binary_metrics(
    massive_pred,"MASSIVE","language",
    ["intent_correct","slot_pair_f1","slot_label_f1","frame_exact",
     "all_pred_values_surface_grounded","hallucinated_slot_value_rate"]
)
add_binary_metrics(
    mtop_pred,"MTOP","language",
    ["intent_correct","slot_pair_f1","slot_label_f1","frame_exact",
     "all_pred_values_surface_grounded","hallucinated_slot_value_rate"]
)
add_binary_metrics(
    mlcl_pred,"MLCL","language",
    ["function_correct","argument_key_exact","shared_argument_value_accuracy","full_tool_call_exact"]
)

metrics_df=pd.DataFrame(metric_rows)
metrics_df.to_csv(RESULTS/"10_external_dataset_metrics.csv",index=False)

display(metrics_df.round(4))

In [ ]:
# CELL 17 — PAPER-READY TABLES

def pct(x):
    return f"{100*x:.2f}%" if pd.notna(x) else "NA"

summary_rows=[]

for dataset,df in [("MASSIVE",massive_pred),("MTOP",mtop_pred),("MLCL",mlcl_pred)]:
    if df.empty: continue

    if dataset in {"MASSIVE","MTOP"}:
        summary_rows.append({
            "Dataset":dataset,
            "N":len(df),
            "Intent accuracy":pct(df["intent_correct"].mean()),
            "Slot pair F1":pct(df["slot_pair_f1"].mean()),
            "Frame exact":pct(df["frame_exact"].mean()),
            "Grounded predicted values":pct(df["all_pred_values_surface_grounded"].mean()),
        })
    else:
        summary_rows.append({
            "Dataset":dataset,
            "N":len(df),
            "Intent accuracy":pct(df["function_correct"].mean()),
            "Slot pair F1":pct(df["shared_argument_value_accuracy"].mean()),
            "Frame exact":pct(df["full_tool_call_exact"].mean()),
            "Grounded predicted values":"n/a",
        })

summary_df=pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS/"11_paper_external_summary.csv",index=False)
display(summary_df)

# LaTeX table
latex = summary_df.to_latex(index=False, escape=True)
(RESULTS/"11_paper_external_summary.tex").write_text(latex,encoding="utf-8")
print(latex)

In [ ]:
# CELL 18 — FIGURES

# Figure A: headline exact/accuracy metrics
plot_rows=[]
for dataset,df in [("MASSIVE",massive_pred),("MTOP",mtop_pred),("MLCL",mlcl_pred)]:
    if df.empty: continue
    if dataset in {"MASSIVE","MTOP"}:
        plot_rows += [
            {"dataset":dataset,"metric":"Intent / function","value":df["intent_correct"].mean()},
            {"dataset":dataset,"metric":"Slot / argument","value":df["slot_pair_f1"].mean()},
            {"dataset":dataset,"metric":"Exact frame / call","value":df["frame_exact"].mean()},
        ]
    else:
        plot_rows += [
            {"dataset":dataset,"metric":"Intent / function","value":df["function_correct"].mean()},
            {"dataset":dataset,"metric":"Slot / argument","value":df["shared_argument_value_accuracy"].mean()},
            {"dataset":dataset,"metric":"Exact frame / call","value":df["full_tool_call_exact"].mean()},
        ]

plot_df=pd.DataFrame(plot_rows)

if len(plot_df):
    pivot=plot_df.pivot(index="dataset",columns="metric",values="value")
    ax=pivot.plot(kind="bar",figsize=(9,5))
    ax.set_ylim(0,1)
    ax.set_ylabel("Score")
    ax.set_title("External dataset generalization")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURES/"external_dataset_generalization.png",dpi=300,bbox_inches="tight")
    plt.show()

# Figure B: language-wise exact metrics
lang_rows=[]
for dataset,df,metric in [
    ("MASSIVE",massive_pred,"frame_exact"),
    ("MTOP",mtop_pred,"frame_exact"),
    ("MLCL",mlcl_pred,"full_tool_call_exact"),
]:
    if df.empty: continue
    for lang,g in df.groupby("language"):
        lang_rows.append({"dataset":dataset,"language":lang,"value":g[metric].mean(),"n":len(g)})

lang_df=pd.DataFrame(lang_rows)
lang_df.to_csv(RESULTS/"12_language_breakdown.csv",index=False)

if len(lang_df):
    for dataset,g in lang_df.groupby("dataset"):
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.bar(g["language"],g["value"])
        ax.set_ylim(0,1)
        ax.set_ylabel("Exact frame / tool-call score")
        ax.set_title(f"{dataset}: language-wise external performance")
        plt.xticks(rotation=30,ha="right")
        plt.tight_layout()
        plt.savefig(FIGURES/f"{dataset.lower()}_language_breakdown.png",dpi=300,bbox_inches="tight")
        plt.show()

In [ ]:
# CELL 19 — CLAIM GATE + PAPER-READY INTERPRETATION

claim_rows=[]

def claim(name,status,evidence,note):
    claim_rows.append({"claim":name,"status":status,"evidence":evidence,"note":note})

if len(massive_pred):
    claim(
        "Independent natural multilingual NLU evidence",
        "SUPPORTED",
        f"MASSIVE n={len(massive_pred)}",
        "Independent multilingual intent/slot dataset; does not directly test authorization."
    )
else:
    claim("Independent natural multilingual NLU evidence","MISSING","MASSIVE","No valid result rows.")

if len(mtop_pred):
    claim(
        "Independent multilingual semantic parsing evidence",
        "SUPPORTED",
        f"MTOP n={len(mtop_pred)}",
        "Independent task-oriented intent/slot dataset; does not directly test authorization."
    )
else:
    claim("Independent multilingual semantic parsing evidence","MISSING","MTOP","No valid result rows.")

if len(mlcl_pred):
    claim(
        "Independent multilingual tool-calling evidence",
        "SUPPORTED",
        f"MLCL n={len(mlcl_pred)}",
        "Published ACL 2026 multilingual tool-calling diagnostic benchmark."
    )
else:
    claim(
        "Independent multilingual tool-calling evidence",
        "MISSING",
        "MLCL",
        "MLCL repository was unavailable or schema mapping was not resolved."
    )

claim(
    "External end-to-end NiyamTrace-X authorization safety",
    "NOT ESTABLISHED BY THIS NOTEBOOK",
    "External datasets above",
    "These datasets validate semantic/tool-interface generalization, not the full NiyamTrace-X authorization/execution stack."
)

claims_df=pd.DataFrame(claim_rows)
claims_df.to_csv(RESULTS/"20_claim_gate.csv",index=False)
display(claims_df)

paper_lines=[
    "# Paper-ready external validation summary",
    "",
    "Use only after checking the generated CSVs and raw predictions.",
    "",
]
for dataset,df in [("MASSIVE",massive_pred),("MTOP",mtop_pred),("MLCL",mlcl_pred)]:
    if df.empty: continue
    if dataset in {"MASSIVE","MTOP"}:
        paper_lines.append(
            f"- {dataset}: n={len(df)}; intent accuracy={df['intent_correct'].mean():.3f}; "
            f"slot-pair F1={df['slot_pair_f1'].mean():.3f}; "
            f"exact frame={df['frame_exact'].mean():.3f}; "
            f"surface-grounded predicted values={df['all_pred_values_surface_grounded'].mean():.3f}."
        )
    else:
        paper_lines.append(
            f"- MLCL: n={len(df)}; function accuracy={df['function_correct'].mean():.3f}; "
            f"argument-key exact={df['argument_key_exact'].mean():.3f}; "
            f"argument-value accuracy={df['shared_argument_value_accuracy'].mean():.3f}; "
            f"full tool-call exact={df['full_tool_call_exact'].mean():.3f}."
        )

paper_lines += [
    "",
    "Recommended manuscript wording:",
    "",
    "> To test whether the semantic/tool-interface findings extend beyond our generated holdout, "
    "we evaluated the same model backend on independently released multilingual benchmarks. "
    "MASSIVE and MTOP probe natural task-oriented intent/slot generalization, while MLCL probes "
    "multilingual tool-call realization under a fixed execution interface. These external datasets "
    "do not themselves establish end-to-end NiyamTrace-X authorization safety, so we report them "
    "as independent semantic and tool-interface evidence rather than as a replacement for the "
    "frozen authorization benchmark.",
]

(RESULTS/"21_paper_ready_summary.md").write_text("\n".join(paper_lines),encoding="utf-8")
print("\n".join(paper_lines))

In [ ]:
# CELL 20 — REPRODUCIBILITY MANIFEST + EXPORT ZIP

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

manifest={
    "created_utc":pd.Timestamp.utcnow().isoformat(),
    "mode":MODE,
    "seed":SEED,
    "backend":BACKEND,
    "local_model":LOCAL_MODEL if BACKEND=="LOCAL_QWEN" else None,
    "openai_model":OPENAI_MODEL if BACKEND=="OPENAI_COMPATIBLE" else None,
    "limits":LIMITS,
    "datasets":{
        "MASSIVE":"AmazonScience/massive",
        "MTOP":"tasksource/mtop",
        "MLCL":"ACL 2026 Lost in Execution artifact" if mlcl_ready else "not available",
    },
    "row_counts":{
        "MASSIVE":len(massive_pred),
        "MTOP":len(mtop_pred),
        "MLCL":len(mlcl_pred),
    },
}
(RESULTS/"MANIFEST.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

# Copy key notebook outputs into export directory
for folder in [RESULTS,FIGURES,RAW,DATA]:
    if folder.exists():
        target=EXPORT/folder.name
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(folder,target)

zip_path=BASE/"NTX_EXTERNAL_DATASET_VALIDATION_RESULTS.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in EXPORT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(EXPORT)))

print("ZIP:",zip_path)
print("SHA256:",sha256_file(zip_path))
print("Size MiB:",round(zip_path.stat().st_size/(1024**2),2))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## How to use the results in the Scientific Reports paper

Do **not** replace the existing internal 2,000-case experiment.

Instead, add a short external-validation subsection:

- **Internal controlled evidence:** NiyamTrace-X 2,000-case frozen group-disjoint holdout.
- **Independent natural-language evidence:** MASSIVE and/or MTOP.
- **Independent multilingual tool-calling evidence:** MLCL, if the artifact loads successfully.

For the main paper, one compact table with MASSIVE + MLCL headline metrics is enough. Move the full per-language breakdown, MTOP extension, bootstrap confidence intervals, and error analysis to Supplementary Information.

Only claim what the exported `20_claim_gate.csv` supports.